# literaturesearch -- try it

Two things you can run here:

1. **`bibcheck`** catches a fabricated citation in a `.bib` file. Offline in under a
   second; a few seconds with live verification. No keys needed.
2. **A seeded literature search** -- give it one paper you trust, and it walks that paper's
   citation graph, validates everything against Crossref, and writes a clean `.bib`.
   About 25 seconds.

Everything here is deterministic Python. The two stages that need a language model --
screening abstracts for relevance, and pulling numbers out of PDFs with a mandatory
verbatim quote -- are *not* run here; see the last section.

> **On API keys:** OpenAlex has metered its API since February 2026, but the search below
> costs about `$0.0025` -- roughly **40 runs a day on the keyless budget**, so you do not
> need a key to try this. For a real search (hundreds of works, several rounds) get a free
> one at <https://openalex.org/settings/api> and `export OPENALEX_API_KEY=...`; it raises
> the budget tenfold. If the budget ever does run out, the run says so and names the cause
> rather than failing silently. `bibcheck` needs no key at all.

## Setup

In [1]:
import os
import pathlib
import subprocess
import sys
import textwrap

# The notebook lives in examples/<name>/, so the repo root is two levels up. __file__ does
# not exist in a notebook, hence the VSCode-specific global with a cwd fallback.
_nb = globals().get("__vsc_ipynb_file__", "")
HERE = pathlib.Path(_nb).resolve().parent if _nb else pathlib.Path.cwd()
REPO = HERE.parents[1] if (HERE.parents[1] / "litsearch").is_dir() else HERE
sys.path.insert(0, str(REPO))

WORK = REPO / "out" / "notebook_demo"      # gitignored; nothing is written to the repo
WORK.mkdir(parents=True, exist_ok=True)

import bibcheck
import litsearch

print("repo      :", REPO)
print("bibcheck  :", pathlib.Path(bibcheck.__file__).parent)
print("scratch   :", WORK)
print("openalex key:", "set" if os.environ.get("OPENALEX_API_KEY") else "none (fine for this demo)")

repo      : C:\Users\lukassp\Documents\conda_envs\msmt202q\literaturesearch
bibcheck  : C:\Users\lukassp\Documents\conda_envs\msmt202q\literaturesearch\bibcheck
scratch   : C:\Users\lukassp\Documents\conda_envs\msmt202q\literaturesearch\out\notebook_demo
openalex key set: False


---

## 1. Catching a fabricated citation

Two entries below. The first is real. The second I invented -- plausible authors, plausible
title, and a **real DOI that belongs to a completely different paper**. That is exactly what
a hallucinated citation looks like in the wild, and skimming a reference list will not catch
it.

`bibcheck` does not merely check that the DOI resolves. It checks that it resolves to *the
paper you claimed*.

In [2]:
demo_bib = WORK / "refs.bib"
demo_bib.write_text(textwrap.dedent("""\
    @article{koch2007,
      author  = {Koch, Jens and Yu, Terri M.},
      title   = {Charge-insensitive qubit design derived from the Cooper pair box},
      journal = {Physical Review A},
      year    = {2007},
      doi     = {10.1103/PhysRevA.76.042319}
    }
    @article{zhang2022,
      author  = {Zhang, Wei and Kumar, Rajesh},
      title   = {Ultrafast parametric entanglement in superconducting cavity arrays},
      journal = {Physical Review Letters},
      year    = {2022},
      doi     = {10.1103/PhysRevLett.128.180502}
    }
"""), encoding="utf-8")

print(demo_bib.read_text(encoding="utf-8"))

@article{koch2007,
  author  = {Koch, Jens and Yu, Terri M.},
  title   = {Charge-insensitive qubit design derived from the Cooper pair box},
  journal = {Physical Review A},
  year    = {2007},
  doi     = {10.1103/PhysRevA.76.042319}
}
@article{zhang2022,
  author  = {Zhang, Wei and Kumar, Rajesh},
  title   = {Ultrafast parametric entanglement in superconducting cavity arrays},
  journal = {Physical Review Letters},
  year    = {2022},
  doi     = {10.1103/PhysRevLett.128.180502}
}



### Offline first

No network. Structural checks only: key style, missing fields, duplicates, DOI shape.
`--no-write` means it reports and touches nothing.

In [3]:
def bibcheck_run(*args):
    """Run the bibcheck CLI and show its output."""
    result = subprocess.run(
        [sys.executable, "-m", "bibcheck.main", *map(str, args)],
        capture_output=True, text=True, cwd=REPO,
        env={**os.environ, "PYTHONIOENCODING": "utf-8"},
    )
    print(result.stdout or result.stderr)
    print(f"[exit code {result.returncode}]   0 = clean, 1 = warnings, 2 = errors")
    return result

_ = bibcheck_run(demo_bib, "--no-write")

bibcheck: C:\Users\lukassp\Documents\conda_envs\msmt202q\literaturesearch\out\notebook_demo\refs.bib
  entries      : 2
  keys renamed : 2
  errors       : 0
  warnings     : 4

Key renames (no .tex files are touched; apply these yourself):
  koch2007  -> Koch2007
  zhang2022 -> Zhang2022

Findings:
  [warning] Koch2007: @article has no volume
  [warning] Koch2007: @article has no pages
  [warning] Zhang2022: @article has no volume
  [warning] Zhang2022: @article has no pages

[exit code 1]   0 = clean, 1 = warnings, 2 = errors


### Now with verification

This one goes to Crossref, DataCite and arXiv. Watch the `Zhang2022` line: the DOI is real,
but it belongs to a paper about quantum key distribution over optical fibre. The citation is
fabricated, and the tool says so and shows you what the DOI actually points at.

Takes a few seconds -- requests are throttled to one per second to stay polite with
Crossref.

In [4]:
_ = bibcheck_run(demo_bib, "--verify", "--no-write")

bibcheck: C:\Users\lukassp\Documents\conda_envs\msmt202q\literaturesearch\out\notebook_demo\refs.bib
  entries      : 2
  keys renamed : 2
  verification : mismatched=1, verified=1
  errors       : 1
  warnings     : 4

Key renames (no .tex files are touched; apply these yourself):
  koch2007  -> Koch2007
  zhang2022 -> Zhang2022

Findings:
  [error  ] Zhang2022: title: local 'Ultrafast parametric entanglement in superconducting cavity arrays' != crossref 'Quantum Key Distribution over 658 km Fiber with Distributed Vibration Sensing'
  [warning] Koch2007: @article has no volume
  [warning] Koch2007: @article has no pages
  [warning] Zhang2022: @article has no volume
  [warning] Zhang2022: @article has no pages

[exit code 2]   0 = clean, 1 = warnings, 2 = errors


**Try it on your own bibliography.** Point it at a real `.bib` and see what comes back:

```python
bibcheck_run("/path/to/your/refs.bib", "--verify", "--no-write")
```

Nothing is modified. With `--out-dir` it writes a cleaned *copy* -- keys unified to
`LastnameYEAR`, entries sorted, fields in canonical order -- and leaves your original alone.

Verification is one request per entry, so a 50-entry bibliography takes a few minutes cold.
Responses are cached, so a second run is nearly instant.

---

## 2. A seeded literature search

Give it **one paper you already trust**. It resolves that DOI, walks its references and
citers, filters what is off topic, and validates every survivor against Crossref before
anything reaches the output.

The seed here is Chapman et al., *High-On-Off-Ratio Beam-Splitter Interaction for Gates on
Bosonically Encoded Qubits*, PRX Quantum **4**, 020355 (2023).

Deliberately small so it finishes in about 25 seconds. Turn `max_rounds`, `seeds_per_round`
and `refs_per_seed` up to go wider -- and note that expansion is throttled to one request
per second, so roughly `seeds_per_round * (1 + refs_per_seed)` seconds per round.

In [ ]:
# Outputs go to ~/litsearch-runs/<name>/ by default -- outside the repository, because
# run results are data. Set LITSEARCH_OUT_DIR to put them somewhere else.
from litsearch.pipeline import SearchSpec, run

SPEC = SearchSpec(
    name="notebook_demo",
    question="What other parametric gates exist for bosonic cavities?",
    seed_dois=("10.1103/PRXQuantum.4.020355",),          # <-- your seed paper
    queries=["parametric beam splitter bosonic cavity microwave"],
    year_from=2015,
    per_query_limit=25,
    max_rounds=1,
    seeds_per_round=3,
    refs_per_seed=4,
    screen_required=("superconduct", "transmon", "bosonic", "microwave cavity", "josephson"),
    screen_forbidden=("optomechanic", "nitrogen-vacancy", "trapped ion"),
    extraction_schema=("gate_type", "coupler", "fidelity_pct"),
)

exit_code = run(SPEC)
print("\nexit code:", exit_code, " (1 means a known-item was missed; none configured here)")

### What it produced

Only works that passed the validation gate reach `refs.bib`. Anything unresolved goes to
`quarantine.md` with a reason -- reported, never silently dropped and never silently
included.

In [ ]:
from litsearch.config import run_dir

out = run_dir("notebook_demo")
print("outputs in:", out)
print()
for path in sorted(out.iterdir()):
    if path.is_file():
        print(f"{path.stat().st_size / 1024:8.1f} KB  {path.name}")

print("\n--- refs.bib, first entry ---")
print("\n".join((out / "refs.bib").read_text(encoding="utf-8").splitlines()[:12]))

In [ ]:
import json

log = json.loads((out / "run.json").read_text(encoding="utf-8"))
print("corpus     :", log["corpus_size"])
print("verified   :", log["verified"])
print("quarantined:", log["quarantined"])
print("\nsnowball rounds (new_fraction should FALL; if it stays high the search stopped early):")
for r in log["saturation_curve"]:
    print(f"  round {r['round']}: {r['new']:4d} new, {r['off_topic_dropped']:4d} dropped as off topic, "
          f"new_fraction {r['new_fraction']}")

### The shortlist, and why it looks the way it does

`shortlist.md` ranks everything that passed the *validation* gate by citation count. It has
**not** been screened for relevance yet, so the top entries are whatever famous, heavily
cited papers the citation graph brushed against -- here, reviews of topological photonics
and circuit QED.

That is not a bug, it is the argument for stage 4. A keyword guard cannot tell a review of
photonics from a paper reporting a parametric gate; both are full of the words "cavity",
"parametric" and "mode". Only reading the abstract against your criteria can, which is
exactly what the screening batches below are for.

In [ ]:
print((out / "shortlist.md").read_text(encoding="utf-8")[:1400])

---

## 3. What is deliberately not here

Two stages need a language model, and this notebook does not call one:

- **Screening** -- reading each abstract against your inclusion criteria. The run above
  wrote its batches to `screen/batch_*.json` and stopped.
- **Extraction** -- reading the PDFs and pulling out the numbers you asked for, each with
  the verbatim sentence it came from. Tasks are in `extract/task_*.json`.

They hand off through files, so a full search is three invocations of the same script with
an agent answering in between. Inside Claude Code, `/litsearch "<question>"` drives all
three. The design keeps every model call outside the deterministic code, which is what
makes the stages inspectable, re-runnable, and correctable by hand.

Have a look at what a screening batch actually asks for:

In [ ]:
batch = sorted((out / "screen").glob("batch_*.json"))[0]
payload = json.loads(batch.read_text(encoding="utf-8"))
print("instructions:", payload["instructions"][:300], "...\n")
print("works in this batch:", len(payload["works"]))
first = payload["works"][0]
print("\none work, as the screener sees it:")
for key, label in [("i", "corpus index"), ("c", "checksum"), ("y", "year"), ("t", "title")]:
    if key in first:
        print(f"  {label:13s} {str(first[key])[:70]}")
print(f"  {'abstract':13s} {str(first.get('a', ''))[:200]}...")

The `c` field is a checksum the screener must echo back. It exists because a verdict
applied to the wrong paper produces a clean-looking, fully validated bibliography of papers
nobody actually screened -- which happened once, and is undetectable downstream.

---

## Where to go next

- [`examples/parametric_gates_bosonic_cavities/README.md`](README.md) -- the full worked
  search, with the numbers from a completed run and an honest list of what is unfinished.
- [`run_search.py`](../../run_search.py) in the repo root -- a commented template to copy.
- [`docs/OUTPUT_FORMATS.md`](../../docs/OUTPUT_FORMATS.md) -- the rules every output obeys:
  every claim carries a cite key, every number traces to a quote.
- `python benchmark/benchmark.py` -- the regression benchmark for the deterministic layers.